# Query Optimization & Explain Plans (Day 10)

## Objective
Learn how to analyze and optimize Spark query performance using:
* **`.explain()`**: Understand query execution plans
* **Partition pruning**: Skip unnecessary data scanning
* **Caching**: Store frequently accessed data in memory
* **Performance benchmarking**: Measure query execution time

## What You'll Learn
1. Reading Spark execution plans (Physical & Logical)
2. Identifying performance bottlenecks
3. Applying optimization techniques
4. Measuring impact of optimizations

## Dataset
* **Table**: `h2_gold_model_scoring_base` (17,535 hourly energy records)
* **Partitioning**: Can partition by `year`, `zone`, or `event_time_utc`
* **Use Case**: Optimize queries for production ML pipeline

## Challenge Tasks (Day 10)
✅ Run heavy query  
✅ Analyze explain plan  
✅ Enable caching  
✅ Compare execution time

In [0]:
# Configuration
CATALOG = "dbacademy"
SCHEMA = "labuser13955035_1772876383"
BASE_TABLE = f"{CATALOG}.{SCHEMA}.h2_gold_model_scoring_base"

print(f"📊 Base Table: {BASE_TABLE}")
print(f"🎯 Optimization Target: Energy price queries")
print("="*80)

# Load table
df = spark.table(BASE_TABLE)
print(f"\n✅ Loaded {df.count():,} records")
print(f"📅 Date Range: {df.select('event_time_utc').agg({'event_time_utc': 'min'}).collect()[0][0]} to {df.select('event_time_utc').agg({'event_time_utc': 'max'}).collect()[0][0]}")

## What is `.explain()`?

Spark's `.explain()` method shows the **execution plan** for a DataFrame query. It reveals:
* **Logical Plan**: What you want to compute (high-level)
* **Optimized Logical Plan**: After Catalyst optimizer rewrites
* **Physical Plan**: How Spark will actually execute (low-level)

### Explain Modes
```python
df.explain(True)   # Show all plans (verbose)
df.explain(False)  # Show physical plan only (default)
df.explain('formatted')  # Pretty-printed with details
```

### Key Terms in Explain Plans
* **FileScan**: Reading data from storage (S3, Delta)
* **Filter**: WHERE clause predicates
* **Project**: SELECT column selection
* **Exchange**: Data shuffle between partitions
* **Sort**: ORDER BY operations
* **Aggregate**: GROUP BY operations
* **PartitionFilters**: Partition pruning (very fast!)
* **PushedFilters**: Predicate pushdown to storage

In [0]:
# Query: Find high-price periods (> 100 EUR/MWh)
print("🔍 Query: SELECT * FROM table WHERE price_eur_mwh > 100")
print("="*80)

high_price_df = df.filter("price_eur_mwh > 100")

# Show explain plan
print("\n📋 PHYSICAL PLAN:")
high_price_df.explain(mode="simple")

print("\n📋 FULL EXPLAIN (All Stages):")
high_price_df.explain(mode="extended")

## Partition Pruning

**What**: Skip reading entire partitions that don't match query filters  
**Benefit**: Dramatically reduce data scanned (10-100x speedup)  
**How**: Partition tables by frequently filtered columns (date, zone, category)

### Example
If table is partitioned by `year`:
```python
# Without partition pruning (scans ALL data)
SELECT * FROM table WHERE price > 100

# With partition pruning (scans only 2021 data)
SELECT * FROM table WHERE year = 2021 AND price > 100
```

### Best Practices
* Partition by **low-cardinality columns** (year, month, zone)
* Avoid **high-cardinality partitions** (user_id, timestamp)
* Target **100MB - 1GB per partition** for optimal performance

### How to Check
Look for `PartitionFilters` in explain plan:
```
PartitionFilters: [isnotnull(year#123), (year#123 = 2021)]
```

In [0]:
import time

# Scenario: Query 2021 data only
print("⚡ PARTITION PRUNING COMPARISON")
print("="*80)

# Without partition filter (scans all years)
print("\n❌ BAD: Filter without partition column")
start = time.time()
bad_query = df.filter("price_eur_mwh > 50").count()
elapsed_bad = time.time() - start
print(f"Result: {bad_query:,} records")
print(f"Time: {elapsed_bad:.2f}s")

# With partition filter (scans only 2021)
print("\n✅ GOOD: Filter with partition column (year extracted from event_time_utc)")
from pyspark.sql.functions import year as spark_year
start = time.time()
good_query = df.filter((spark_year("event_time_utc") == 2021) & (df.price_eur_mwh > 50)).count()
elapsed_good = time.time() - start
print(f"Result: {good_query:,} records")
print(f"Time: {elapsed_good:.2f}s")

# Speedup
speedup = elapsed_bad / elapsed_good if elapsed_good > 0 else 1
print(f"\n🚀 Speedup: {speedup:.2f}x faster")
print(f"💡 Tip: Partition tables by year or month for even better pruning")

## DataFrame Caching

**What**: Store DataFrame in memory/disk for reuse  
**Benefit**: Avoid re-reading data from storage (10-100x speedup for repeated queries)  
**Cost**: Memory usage (monitor cluster memory)

### When to Cache
✅ **Iterative algorithms** (ML training)  
✅ **Repeated queries** on same dataset  
✅ **Interactive analysis** (Jupyter notebooks)  
✅ **Small-medium datasets** (< cluster memory)

### When NOT to Cache
❌ **Large datasets** (> cluster memory)  
❌ **One-time queries** (caching overhead not worth it)  
❌ **Streaming data** (data changes constantly)

### Cache Methods
```python
df.cache()           # Memory only (default)
df.persist()         # Memory + Disk spillover
df.unpersist()       # Remove from cache
```

### Cache Levels
* `MEMORY_ONLY`: Fast but limited by RAM
* `MEMORY_AND_DISK`: Spills to disk if needed
* `DISK_ONLY`: Slower but no memory pressure

In [0]:
import time

print("💾 CACHING PERFORMANCE COMPARISON")
print("="*80)

# Load fresh DataFrame (uncached)
df_uncached = spark.table(BASE_TABLE)

# Test 1: Uncached query (reads from storage)
print("\n❌ UNCACHED: Query 1 (reads from Delta storage)")
start = time.time()
result1_uncached = df_uncached.filter("price_eur_mwh > 50").count()
time1_uncached = time.time() - start
print(f"Result: {result1_uncached:,} records")
print(f"Time: {time1_uncached:.3f}s")

print("\n❌ UNCACHED: Query 2 (reads from Delta storage AGAIN)")
start = time.time()
result2_uncached = df_uncached.filter("price_eur_mwh > 100").count()
time2_uncached = time.time() - start
print(f"Result: {result2_uncached:,} records")
print(f"Time: {time2_uncached:.3f}s")

# Test 2: Cached query (reads from memory)
print("\n" + "="*80)
print("✅ CACHED: Enabling cache...")
df_cached = spark.table(BASE_TABLE).cache()
_ = df_cached.count()  # Materialize cache
print("Cache populated!")

print("\n✅ CACHED: Query 1 (reads from memory)")
start = time.time()
result1_cached = df_cached.filter("price_eur_mwh > 50").count()
time1_cached = time.time() - start
print(f"Result: {result1_cached:,} records")
print(f"Time: {time1_cached:.3f}s")

print("\n✅ CACHED: Query 2 (reads from memory)")
start = time.time()
result2_cached = df_cached.filter("price_eur_mwh > 100").count()
time2_cached = time.time() - start
print(f"Result: {result2_cached:,} records")
print(f"Time: {time2_cached:.3f}s")

# Summary
print("\n" + "="*80)
print("📊 PERFORMANCE SUMMARY")
print(f"Uncached Avg: {(time1_uncached + time2_uncached)/2:.3f}s")
print(f"Cached Avg: {(time1_cached + time2_cached)/2:.3f}s")
speedup = ((time1_uncached + time2_uncached) / (time1_cached + time2_cached)) if (time1_cached + time2_cached) > 0 else 1
print(f"🚀 Speedup: {speedup:.2f}x faster with caching")

# Cleanup
df_cached.unpersist()
print("\n🧹 Cache cleared")

## Predicate Pushdown

**What**: Push WHERE filters down to storage layer (Delta, Parquet)  
**Benefit**: Read only necessary data, skip entire files  
**How**: Spark automatically pushes filters to Delta/Parquet readers

### How It Works
```python
# Your query
df.filter("price > 100").select("event_time_utc", "price_eur_mwh")

# What Spark does:
# 1. Push filter to Delta reader (PushedFilters)
# 2. Delta skips files where max(price) <= 100
# 3. Only read matching files
# 4. Return filtered + projected columns
```

### Check Explain Plan
Look for `PushedFilters` in explain output:
```
PushedFilters: [IsNotNull(price_eur_mwh), GreaterThan(price_eur_mwh, 100)]
```

### Best Practices
✅ Use simple filters (=, >, <, IN)  
✅ Filter early in query chain  
✅ Use column statistics (Delta automatic)  
❌ Avoid complex UDFs (prevent pushdown)

In [0]:
print("🔽 PREDICATE PUSHDOWN ANALYSIS")
print("="*80)

# Query with simple filter (pushdown works)
print("\n✅ GOOD: Simple filter (pushdown works)")
simple_filter = df.filter("price_eur_mwh > 100 AND zone = 'NL'")
print("\nExplain Plan:")
simple_filter.explain(mode="formatted")

print("\n" + "="*80)
print("Look for 'PushedFilters' in the FileScan output above")
print("These filters are executed at the storage layer (Delta/Parquet)")
print("Result: Fewer files read, faster query execution")

## Identifying Expensive Operations

In explain plans, look for:

### 🔴 SLOW Operations (Avoid)
* **Exchange (Shuffle)**: Data movement across network
  - Triggered by: GROUP BY, JOIN, DISTINCT, repartition()
  - Cost: Network I/O + serialization
  - Fix: Reduce data before shuffle, use broadcast joins

* **Sort**: Ordering data
  - Triggered by: ORDER BY, sortWithinPartitions()
  - Cost: Memory + CPU
  - Fix: Avoid sorting large datasets, push to final stage

* **CartesianProduct**: Cross join
  - Triggered by: JOIN without condition
  - Cost: N × M records (explodes data)
  - Fix: Always use JOIN conditions

### 🟢 FAST Operations (Preferred)
* **Filter**: WHERE clauses (especially with pushdown)
* **Project**: SELECT columns (column pruning)
* **BroadcastHashJoin**: Small table joins
* **PartitionFilters**: Partition pruning

### Performance Tuning Checklist
✅ Filters pushed to storage layer?  
✅ Partitions pruned?  
✅ Joins using broadcast for small tables?  
✅ Unnecessary columns dropped early?  
✅ Shuffles minimized?  
✅ Data cached if reused?

In [0]:
import time
from pyspark.sql.functions import col, avg, count, year as spark_year, month

print("⚡ HEAVY QUERY OPTIMIZATION")
print("="*80)

# Heavy query: Aggregate by year and month
print("\n❌ BAD: Expensive aggregation without optimization")
start = time.time()
bad_query = df \
    .groupBy(spark_year("event_time_utc").alias("year"), month("event_time_utc").alias("month")) \
    .agg(
        avg("price_eur_mwh").alias("avg_price"),
        count("*").alias("record_count")
    ) \
    .orderBy("year", "month")

print("\n📋 Explain Plan (Bad Query):")
bad_query.explain(mode="simple")

result_bad = bad_query.collect()
time_bad = time.time() - start
print(f"\nResult: {len(result_bad)} rows")
print(f"Time: {time_bad:.3f}s")

# Optimized query: Filter first, then aggregate
print("\n" + "="*80)
print("✅ GOOD: Optimized with filtering and caching")
start = time.time()

# Step 1: Filter to reduce data
filtered_df = df.filter(spark_year("event_time_utc").isin([2020, 2021]))

# Step 2: Cache filtered data (if reused)
filtered_df.cache().count()

# Step 3: Aggregate
good_query = filtered_df \
    .groupBy(spark_year("event_time_utc").alias("year"), month("event_time_utc").alias("month")) \
    .agg(
        avg("price_eur_mwh").alias("avg_price"),
        count("*").alias("record_count")
    ) \
    .orderBy("year", "month")

result_good = good_query.collect()
time_good = time.time() - start
print(f"\nResult: {len(result_good)} rows")
print(f"Time: {time_good:.3f}s")

# Summary
speedup = time_bad / time_good if time_good > 0 else 1
print(f"\n🚀 Optimization Impact: {speedup:.2f}x faster")

# Display results
import pandas as pd
print("\n📊 Monthly Average Prices:")
display(pd.DataFrame(result_good))

# Cleanup
filtered_df.unpersist()

## ✅ Query Optimization Checklist for Production

### Data Organization
- [ ] **Partition tables** by frequently filtered columns (date, category)
- [ ] **Z-order** Delta tables for multi-column filters
- [ ] **Optimize** Delta tables regularly (OPTIMIZE command)
- [ ] **Vacuum** old files to reduce metadata overhead

### Query Design
- [ ] **Filter early** in query chain
- [ ] **Drop unused columns** with `.select()`
- [ ] **Use broadcast joins** for small tables (< 10MB)
- [ ] **Avoid shuffles** when possible (especially for large data)
- [ ] **Cache DataFrames** that are reused multiple times

### Monitoring
- [ ] **Review explain plans** for expensive operations
- [ ] **Check Spark UI** for task execution times
- [ ] **Monitor cache hit ratio** in Spark UI
- [ ] **Profile slow queries** with Spark SQL metrics

### Cluster Configuration
- [ ] **Right-size cluster** (not too small, not too large)
- [ ] **Enable adaptive query execution** (Spark 3.0+)
- [ ] **Tune shuffle partitions** (spark.sql.shuffle.partitions)
- [ ] **Configure memory** appropriately

### Delta Lake Specific
- [ ] **Enable data skipping** (automatic in Delta)
- [ ] **Use column statistics** (ANALYZE TABLE)
- [ ] **Leverage file pruning** with filters
- [ ] **Check Delta cache** for hot data

In [0]:
print("📊 QUERY OPTIMIZATION SUMMARY (Day 10)")
print("="*80)

print("""
✅ COMPLETED CHALLENGE TASKS:
1. ✅ Run heavy query (aggregation with GROUP BY)
2. ✅ Analyze explain plan (identified shuffles, sorts, filters)
3. ✅ Enable caching (demonstrated cache vs uncached)
4. ✅ Compare execution time (measured speedup)

🎯 KEY LEARNINGS:

1. EXPLAIN PLANS
   • Physical plan shows actual execution steps
   • Look for PushedFilters, PartitionFilters
   • Identify expensive operations (Exchange, Sort)

2. PARTITION PRUNING
   • Skip entire partitions that don't match filters
   • Partition by low-cardinality columns (year, zone)
   • Can achieve 10-100x speedup

3. CACHING
   • Store frequently accessed data in memory
   • Use for iterative algorithms and repeated queries
   • Monitor memory usage in Spark UI

4. PREDICATE PUSHDOWN
   • Automatic filter pushdown to storage layer
   • Delta Lake skips files based on statistics
   • Works best with simple filters (=, >, <)

💡 PRODUCTION RECOMMENDATIONS:

1. Always review explain plans for production queries
2. Partition Delta tables by date (year/month)
3. Cache DataFrames used in ML training loops
4. Monitor Spark UI for bottlenecks
5. Regularly OPTIMIZE and VACUUM Delta tables

🔗 NEXT STEPS:
• Day 11: Time Travel & Data Recovery
• Day 12: Cost Optimization Strategies
• Day 13: Architecture Design
""")

print("\n✅ Day 10 Complete!")